<a href="https://colab.research.google.com/github/sikha-ai/paralleldots-ML-Assignment/blob/main/pipline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
pip install ultralytics easyocr transformers torch torchvision opencv-python-headless scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 21.0 MB/s eta 0:00:00


In [10]:
import cv2
import json
import numpy as np
import torch
import easyocr
from PIL import Image
from ultralytics import YOLO
from sklearn.cluster import DBSCAN
from transformers import CLIPProcessor, CLIPModel

class UniqueShelfPipeline:
    def __init__(self):
        print("[System] Initializing Intelligent Shelf Diagnostics Pipeline...")
        # 1. Product Bounding Box Locator (SOTA Object Detection)
        self.detector = YOLO("yolov8x.pt")

        # 2. Semantic Brand Fingerprinter (Zero-Shot CLIP Classifier)
        self.clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
        self.clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

        # 3. Scene Text Extraction Engine
        self.ocr_reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())

        # Exact target brands requested by the prompt
        self.brand_labels = ["Coca-Cola", "Pepsi", "Lay's chips", "Doritos", "juice bottle", "milk product"]

    def run_zero_shot_classification(self, crop_img):
        """Classifies product patches into brands dynamically using text matching."""
        pil_img = Image.fromarray(cv2.cvtColor(crop_img, cv2.COLOR_BGR2RGB))
        inputs = self.clip_processor(text=self.brand_labels, images=pil_img, return_tensors="pt", padding=True)

        with torch.no_grad():
            outputs = self.clip_model(**inputs)

        probs = outputs.logits_per_image.softmax(dim=-1)
        top_idx = probs.argmax().item()
        matched_label = self.brand_labels[top_idx]

        # Map semantic labels to the precise evaluation target classes
        if "Coca-Cola" in matched_label: return "Coca-Cola"
        if "Pepsi" in matched_label: return "Pepsi"
        if "Lay's" in matched_label: return "Lay's"
        if "Doritos" in matched_label: return "Doritos"
        return "Other"

    def estimate_shelf_rows(self, boxes):
        """Uses 1D DBSCAN clustering on Y-centers to dynamically group items into shelf rows."""
        if len(boxes) == 0: return {}

        # Calculate vertical center points
        y_centers = np.array([(box[1] + box[3]) / 2 for box in boxes]).reshape(-1, 1)

        # Cluster items together if their horizontal centers are within ~10% of total frame variance
        clustering = DBSCAN(eps=45, min_samples=2).fit(y_centers)
        labels = clustering.labels_

        unique_rows = set(labels)
        return {f"Row_{i+1}": int(np.sum(labels == label)) for i, label in enumerate(unique_rows) if label != -1}

    def process_shelf_image(self, image_path):
        image = cv2.imread(image_path)
        if image is None: raise FileNotFoundError(f"Missing file: {image_path}")

        img_name = image_path.split("/")[-1]
        annotated_img = image.copy()

        # Stage 1: Detect all packaging items
        results = self.detector(image, verbose=False)[0]
        detected_boxes = results.boxes.xyxy.cpu().numpy()

        total_products = 0
        brand_counts = {"Coca-Cola": 0, "Pepsi": 0, "Lay's": 0, "Doritos": 0, "Other": 0}
        valid_boxes = []

        # Stage 2: Process localized product patches
        for box in detected_boxes:
            x1, y1, x2, y2 = map(int, box[:4])
            conf = box[4] if len(box) > 4 else 0.85
            if conf < 0.3: continue  # drop noisy false-positives

            crop = image[y1:y2, x1:x2]
            if crop.size == 0: continue

            total_products += 1
            valid_boxes.append([x1, y1, x2, y2])

            # Smart inference classification block
            assigned_brand = self.run_zero_shot_classification(crop)
            brand_counts[assigned_brand] += 1

            # Plot stylized dynamic visual elements
            color = (0, 255, 0) if assigned_brand != "Other" else (255, 120, 0)
            cv2.rectangle(annotated_img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(annotated_img, assigned_brand, (x1, y1 - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1, cv2.LINE_AA)

        # Stage 3: Spatial shelf configuration heuristics
        shelf_rows_breakdown = self.estimate_shelf_rows(valid_boxes)

        # Stage 4: Run OCR text reading pipeline
        ocr_results = self.ocr_reader.readtext(image)
        ocr_labels = []
        for res in ocr_results:
            text = res[1].strip()
            score = res[2]
            # Match characters looking like retail pricing (numbers, currency tags)
            if score > 0.45 and (any(c.isdigit() for c in text) or any(sym in text for sym in ["$", "₹", "for"])) :
                ocr_labels.append(text)

        # Format standardized output payload schema requested by requirements
        output_payload = {
            "image_name": img_name,
            "total_products": total_products,
            "brands": brand_counts,
            "ocr_labels": list(set(ocr_labels))[:10],
            "analytics_meta": {
                "detected_shelf_rows": len(shelf_rows_breakdown),
                "items_per_row_distribution": shelf_rows_breakdown
            }
        }

        return output_payload, annotated_img

In [11]:
import cv2
import json

if __name__ == "__main__":
    # Initialize the pipeline directly from memory
    pipeline = UniqueShelfPipeline()

    # Define the exact names of your uploaded test images
    test_images = ["img_1.jpg", "img_2.jpg", "img_3.jpg"]

    for img in test_images:
        print(f"\n[Processing] Analyzing structural elements in {img}...")
        try:
            metrics, viz_frame = pipeline.process_shelf_image(img)

            # 1. Print structured dictionary response directly to the notebook output
            print(json.dumps(metrics, indent=4))

            # 2. Save file artifacts to disk
            cv2.imwrite(f"annotated_{img}", viz_frame)
            print(f"[Success] Visual bounding plots saved as: annotated_{img}")

        except Exception as e:
            print(f"[Error] Failed to process image {img}: {e}")

[System] Initializing Intelligent Shelf Diagnostics Pipeline...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[Processing] Analyzing structural elements in img_1.jpg...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{
    "image_name": "img_1.jpg",
    "total_products": 54,
    "brands": {
        "Coca-Cola": 9,
        "Pepsi": 12,
        "Lay's": 4,
        "Doritos": 1,
        "Other": 28
    },
    "ocr_labels": [
        "34",
        "40",
        "60",
        "75",
        "50",
        "125",
        "99",
        "30",
        "35",
        "250 m"
    ],
    "analytics_meta": {
        "detected_shelf_rows": 4,
        "items_per_row_distribution": {
            "Row_1": 20,
            "Row_2": 21,
            "Row_3": 11,
            "Row_4": 2
        }
    }
}
[Success] Visual bounding plots saved as: annotated_img_1.jpg

[Processing] Analyzing structural elements in img_2.jpg...
{
    "image_name": "img_2.jpg",
    "total_products": 0,
    "brands": {
        "Coca-Cola": 0,
        "Pepsi": 0,
        "Lay's": 0,
        "Doritos": 0,
        "Other": 0
    },
    "ocr_labels": [
        "106",
        "52 5g",
        "11%",
        "52 $ 9",
        "99",
        "30",
      

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{
    "image_name": "img_3.jpg",
    "total_products": 27,
    "brands": {
        "Coca-Cola": 0,
        "Pepsi": 1,
        "Lay's": 0,
        "Doritos": 0,
        "Other": 26
    },
    "ocr_labels": [
        "4",
        "25",
        "62",
        "45",
        "40",
        "60",
        "52",
        "'140",
        "35",
        "58"
    ],
    "analytics_meta": {
        "detected_shelf_rows": 3,
        "items_per_row_distribution": {
            "Row_1": 15,
            "Row_2": 8,
            "Row_3": 3
        }
    }
}
[Success] Visual bounding plots saved as: annotated_img_3.jpg
